<a href="https://colab.research.google.com/github/efitznm/Capstone-Colab/blob/main/Push_PIT_HIC_excels_to_HF_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import io
import pandas as pd
import requests

In [2]:
# Install pyxlsb to read the .xlsb file format
!pip install -q pyxlsb

In [3]:
# URLs for the data sources
pit_url = "https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-PIT-Counts-by-CoC.xlsb"
hic_url = "https://www.huduser.gov/portal/sites/default/files/xls/2007-2025-HIC-Counts-by-CoC.xlsx"

In [4]:
# Add a browser User-Agent header so huduser.gov allows the download
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        " (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
}

In [7]:
# Fetch both files
pit_response = requests.get(pit_url, headers=headers)
pit_response.raise_for_status()  # Raises an error if the download fails

hic_response = requests.get(hic_url, headers=headers)
hic_response.raise_for_status()

In [8]:
# Check if 'PIT data' successful
print(f"""
{'='*40}
Checking PIT file
{'='*40}
Status: {pit_response.status_code}
Content-Type: {pit_response.headers.get("Content-Type")}
Content size: {len(pit_response.content)}
""")


Checking PIT file
Status: 200
Content-Type: application/vnd.ms-excel.sheet.binary.macroEnabled.12
Content size: 8859426



In [9]:
# Check if 'HIC data' successful
print(f"""
{'='*40}
Checking HIC file
{'='*40}
Status: {hic_response.status_code}
Content-Type: {hic_response.headers.get("Content-Type")}
Content size: {len(hic_response.content)}
""")


Checking HIC file
Status: 200
Content-Type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
Content size: 2598803



In [10]:
# Load the Excel workbooks from memory
pit_excel = pd.ExcelFile(io.BytesIO(pit_response.content), engine="pyxlsb")
hic_excel = pd.ExcelFile(io.BytesIO(hic_response.content))

print("PIT tabs:", pit_excel.sheet_names)
print("HIC tabs:", hic_excel.sheet_names)

PIT tabs: ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', 'Table_categories', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', 'CoC Mergers', 'CoCs, States, DDs', 'field list, all years', 'Generate_Table', 'PIT_Counts_Table_Template', 'PIT_Counts_Table', 'Generate_Chart', 'Chart_categories', 'PIT_Counts_Chart', 'PIT_Counts_Chart_Template']
HIC tabs: ['2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', 'Revisions']


In [11]:
# Save downloaded response bytes to local files in Colab
pit_file = "2007-2025-PIT-Counts-by-CoC.xlsb"
hic_file = "2007-2025-HIC-Counts-by-CoC.xlsx"

with open(pit_file, "wb") as f:
    f.write(pit_response.content)

with open(hic_file, "wb") as f:
    f.write(hic_response.content)

!ls -lh {pit_file} {hic_file}

-rw-r--r-- 1 root root 2.5M Nov 21 01:43 2007-2025-HIC-Counts-by-CoC.xlsx
-rw-r--r-- 1 root root 8.5M Nov 21 01:43 2007-2025-PIT-Counts-by-CoC.xlsb


### Authenticate to Hugging Face


In [12]:
from google.colab import userdata
import os

In [13]:
# Retrieve the token from Colab's secret manager
os.environ["HF_TOKEN"] = userdata.get('hf_token')
_ = os.environ["HF_TOKEN"]
f"{_[:3]} ... {_[-4:]}"

'hf_ ... RmJe'

In [14]:
os.environ["HF_ACCOUNT"] = userdata.get('hf_account')
hf_account = os.environ["HF_ACCOUNT"]
hf_account


'ericbdf7'

In [16]:
hf_repo = "DDDS_Capstone_Housing"
os.environ["HF_REPO"] = hf_repo
hf_repo


'DDDS_Capstone_Housing'

In [17]:
# Log in automatically without an interactive prompt
! hf auth login --token $HF_TOKEN


Hint: A new version of huggingface_hub (2.4.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `colab` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


### Push to Hugging Face


In [20]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  2007-2025-PIT-Counts-by-CoC.xlsb


CalledProcessError: Command 'b'\nhf upload \\\n  --type dataset \\\n  ${HF_ACCOUNT}/${HF_REPO} \\\n  2007-2025-PIT-Counts-by-CoC.xlsb\n'' returned non-zero exit status 1.

In [19]:
print(hf_upload.stdout)


In [ ]:
%%capture hf_upload
%%bash

hf upload \
  --type dataset \
  ${HF_ACCOUNT}/${HF_REPO} \
  2007-2025-HIC-Counts-by-CoC.xlsx


In [ ]:
print(hf_upload.stdout)


✓ Uploaded
  url: https://huggingface.co/datasets/timnickellddds/Data_Science-22/commit/db48dbf76923781eedba252d42d507b3293864ae



## Verify Push to Hugging Face

### Read parquet from Hugging Face


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/transactions.parquet"
hf_url


'https://huggingface.co/datasets/timnickellddds/Data_Science-22/resolve/main/transactions.parquet'

In [ ]:
df3 = pd.read_parquet( hf_url )
df3.shape


(180000, 51)

In [ ]:
df3.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180000 entries, 0 to 179999
Data columns (total 5 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   target  180000 non-null  float64
 1   var_0   180000 non-null  float64
 2   var_1   180000 non-null  float64
 3   var_2   180000 non-null  float64
 4   var_4   180000 non-null  float64
dtypes: float64(5)
memory usage: 6.9 MB


### Read CSV from Hugging Face


In [ ]:
# Construct the base Hugging Face URL for your dataset files
hf_url = f"https://huggingface.co/datasets/{hf_account}/{hf_repo}/resolve/main/transactions.csv"
hf_url


'https://huggingface.co/datasets/timnickellddds/Data_Science-22/resolve/main/transactions.csv'

In [ ]:
df4 = pd.read_csv( hf_url )
df4.shape


(180000, 51)

In [ ]:
df4.iloc[:,:5].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180000 entries, 0 to 179999
Data columns (total 5 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   target  180000 non-null  float64
 1   var_0   180000 non-null  float64
 2   var_1   180000 non-null  float64
 3   var_2   180000 non-null  float64
 4   var_4   180000 non-null  float64
dtypes: float64(5)
memory usage: 6.9 MB
